## CGAN-CNN
This notebook implements a CGAN model for time series. The model generates window_size elements, conditioned on the previous condition_size elements. The discrimator is trained using the Wasserstein loss with gradient penalty. Here 1D convolutions are employed instead of dense layers.

In [ ]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
# tqdm
from tqdm.notebook import tqdm, trange
# matplotlib
import matplotlib.pyplot as plt
# local
from empirical_tests import run_tests, plot_daily_returns, acf_plots_block, qq_plot
from timeseries import download_data, log_returns, normalize, denormalize, TimeSeriesDataset
from metrics import calculate_scores, real_data_loading
from util import create_file_path, plot_losses

In [ ]:
fileprefix = "CGANCNN"

### Load the Data and build the Datasets

In [ ]:
start_date = "2001-01-01"
end_date = "2021-01-01"

 # Single stock example: Apple Inc. (AAPL)
single_stock = download_data("AAPL", start_date=start_date, end_date=end_date)

# Stock index example: S&P 500 Index (^GSPC)
stock_index = download_data("^GSPC", start_date=start_date, end_date=end_date)

# FX spot example: USD/JPY (JPY=X)
fx_spot = download_data("JPY=X", start_date=start_date, end_date=end_date)

# Commodity spot example: Gold (GC=F)
commodity_spot = download_data("GC=F", start_date=start_date, end_date=end_date)

ts_lst = [single_stock['Close'].to_numpy(), stock_index['Close'].to_numpy(), fx_spot['Close'].to_numpy(), commodity_spot['Close'].to_numpy()]

single_stock_returns = log_returns(single_stock['Close'])
stock_index_returns = log_returns(stock_index['Close'])
fx_spot_returns = log_returns(fx_spot['Close'])
commodity_spot_returns = log_returns(commodity_spot['Close'])

lst_returns = [single_stock_returns, stock_index_returns, fx_spot_returns, commodity_spot_returns]
lst_labels = ['Apple', 'S&P 500', 'USDJPY', 'Gold']

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
norm_log_ssr, mlr_ssr, slr_ssr = normalize(single_stock_returns)
norm_log_sir, mlr_sir, slr_sir = normalize(stock_index_returns)
norm_log_fxr, mlr_fxr, slr_fxr = normalize(fx_spot_returns)
norm_log_csr, mlr_csr, slr_csr = normalize(commodity_spot_returns)

In [ ]:
# Convert DataFrame to PyTorch Tensor
ssr_tensor = torch.tensor(norm_log_ssr, dtype=torch.float32)
sir_tensor = torch.tensor(norm_log_sir, dtype=torch.float32)
fxr_tensor = torch.tensor(norm_log_fxr, dtype=torch.float32)
csr_tensor = torch.tensor(norm_log_csr, dtype=torch.float32)

In [ ]:
window_size = 32
condition_size = 32

In [ ]:
# Create TensorDataset and DataLoader
ssr_dataset = TimeSeriesDataset(ssr_tensor, window_size, condition_size)
sir_dataset = TimeSeriesDataset(sir_tensor, window_size, condition_size)
fxr_dataset = TimeSeriesDataset(fxr_tensor, window_size, condition_size)
csr_dataset = TimeSeriesDataset(csr_tensor, window_size, condition_size)

In [ ]:
ssr2 = real_data_loading(single_stock_returns, 32)
sir2 = real_data_loading(stock_index_returns, 32)
fxr2 = real_data_loading(fx_spot_returns, 32)
csr2 = real_data_loading(commodity_spot_returns, 32)
real_lst = [ssr2, sir2, fxr2, csr2]

### Build the Generator and Discriminator

In [ ]:
class Generator(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim):
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, output_dim, kernel_size=3, padding=1),
        )

    def forward(self, x):
        x = x.unsqueeze(2)
        output = self.model(x)
        return output.squeeze(2)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, 1, kernel_size=3, padding=1),
        )

    def forward(self, x):
        x = x.unsqueeze(2)
        output = self.model(x)
        return output.squeeze(2)

### Define a function to calculate the Gradient Penalty term

In [ ]:
def gradient_penalty(critic, real_samples, fake_samples, device):
    alpha = torch.rand(real_samples.shape[0], 1).to(device)
    interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
    d_interpolates = critic(interpolates)
    gradients = torch.autograd.grad(outputs=d_interpolates, inputs=interpolates,
                                     grad_outputs=torch.ones(d_interpolates.size()).to(device),
                                     create_graph=True, retain_graph=True)[0]
    gradients = gradients.view(gradients.size(0), -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

### Define the training loop

In [ ]:
def train_cgan(dataset, window_size, condition_size, epochs,
               batch_size, device,
               savefilepath,
               savefilename,
               lr=1e-4, latent_dim=32,
               hidden_dim=128,
               lambda_gp=10,
               epoch_save=100):

    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Infer condition feature size from one sample
    sample_condition, sample_window = dataset[0]
    cond_flat_dim = sample_condition.numel()   # flattened condition size
    gen_input_dim = cond_flat_dim + latent_dim
    gen_output_dim = sample_window.numel() if hasattr(sample_window, "numel") else window_size
    disc_input_dim = gen_output_dim

    generator = Generator(gen_input_dim, gen_output_dim, hidden_dim).to(device)
    discriminator = Discriminator(disc_input_dim, hidden_dim).to(device)

    optimizer_G = optim.Adam(generator.parameters(), lr=lr)
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr)

    gen_loss_list = []
    disc_loss_list = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        num_items = 0
        gen_loss = 0.0
        disc_loss = 0.0

        for condition, window in dataloader:
            condition = condition.to(device).float()
            window = window.to(device).float()
            cur_batch_size = condition.size(0)

            # Flatten condition and window for linear layers
            condition_flat = condition.reshape(cur_batch_size, -1)
            real_samples = window.reshape(cur_batch_size, -1)

            # Train Discriminator
            optimizer_D.zero_grad()

            z = torch.randn(cur_batch_size, latent_dim, device=device)
            gen_input = torch.cat((condition_flat, z), dim=1)
            fake_samples = generator(gen_input)

            real_validity = discriminator(real_samples)
            fake_validity = discriminator(fake_samples.detach())

            gp = gradient_penalty(discriminator, real_samples, fake_samples.detach(), device)
            d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gp

            disc_loss += d_loss.item() * cur_batch_size
            d_loss.backward()
            optimizer_D.step()

            # Train Generator
            optimizer_G.zero_grad()

            z = torch.randn(cur_batch_size, latent_dim, device=device)
            gen_input = torch.cat((condition_flat, z), dim=1)
            fake_samples = generator(gen_input)
            fake_validity = discriminator(fake_samples)

            g_loss = -torch.mean(fake_validity)
            gen_loss += g_loss.item() * cur_batch_size
            g_loss.backward()
            optimizer_G.step()

            num_items += cur_batch_size

        ave_g_loss = gen_loss / num_items
        ave_d_loss = disc_loss / num_items
        gen_loss_list.append(ave_g_loss)
        disc_loss_list.append(ave_d_loss)

        tqdm_epoch.set_description(
            'Ave Gen Loss: {:5f}, Ave Disc Loss: {:5f}'.format(ave_g_loss, ave_d_loss)
        )

        if epoch % epoch_save == 0:
            fullname = savefilename + '_' + str(epoch) + '.pth'
            fullpath = create_file_path(savefilepath, fullname)
            torch.save(generator.state_dict(), fullpath)

    return generator, discriminator, gen_loss_list, disc_loss_list

In [ ]:
epochs = 1000
batch_size = 64
lr = 1e-5

In [ ]:
def generate_time_series(generator, initial_condition, length, device, latent_dim=32, condition_size=32):
    generator.eval()

    # initial_condition should contain 32 values
    current_condition = initial_condition.detach().clone().float().to(device).view(-1)

    if current_condition.numel() != condition_size:
        raise ValueError(
            f"initial_condition must have {condition_size} values, got {current_condition.numel()}"
        )

    generated = current_condition.clone().cpu().tolist()

    with torch.no_grad():
        while len(generated) < length:
            # shape: [1, 32]
            condition_input = current_condition.unsqueeze(0)

            # shape: [1, 32]
            z = torch.randn(1, latent_dim, device=device)

            # shape: [1, 64]
            gen_input = torch.cat((condition_input, z), dim=1)

            # generator output shape: [1, 32]
            generated_window = generator(gen_input).squeeze(0)

            # take one next value from the generated window
            next_value = generated_window[0]

            generated.append(next_value.item())

            # roll condition window forward
            current_condition = torch.cat(
                (current_condition[1:], next_value.unsqueeze(0)),
                dim=0
            )

    return torch.tensor(generated, dtype=torch.float32)

In [ ]:
sim_returns = list()
savefilepath = fileprefix
loss_names = ['Generator', 'Discriminator']

### Train Single Stock Model

In [ ]:
ssr_savename = 'ssrmodel'
ssr_gen, _, ssrgl, ssrdl = train_cgan(ssr_dataset, window_size, condition_size, 
                              epochs, batch_size, device, savefilepath, ssr_savename, lr)

In [ ]:
plot_losses([ssrgl, ssrdl], loss_names, epochs, fileprefix + ssr_savename + '_loss.png')

In [ ]:
# Define the initial condition and desired generated length
initial_condition = torch.tensor(ssr_tensor[:condition_size], dtype=torch.float32).to(device)
generated_length = len(ssr_tensor)

# Generate the new time series
norm_sample_ssr = generate_time_series(ssr_gen, initial_condition, generated_length, device)

In [ ]:
sample_ssr = denormalize(norm_sample_ssr, mlr_ssr, slr_ssr)
sim_returns.append(sample_ssr[:generated_length])

### Train Stock Index Model

In [ ]:
sir_savename = 'sirmodel'
sir_gen, _, sirgl, sirdl = train_cgan(sir_dataset, window_size, condition_size,
                              epochs, batch_size, device, savefilepath, sir_savename, lr)

In [ ]:
plot_losses([sirgl, sirdl], loss_names, epochs, fileprefix + sir_savename + '_loss.png')

In [ ]:
# Define the initial condition and desired generated length
initial_condition = torch.tensor(sir_tensor[:condition_size], dtype=torch.float32).to(device)

# Generate the new time series
norm_sample_sir = generate_time_series(sir_gen, initial_condition, generated_length, device)

In [ ]:
sample_sir = denormalize(norm_sample_sir, mlr_sir, slr_sir)
sim_returns.append(sample_sir[:generated_length])

### Train FX Model

In [ ]:
fxr_savename = 'fxrmodel'
fxr_gen, _, fxrgl, fxrdl = train_cgan(fxr_dataset, window_size, condition_size, 
                              epochs, batch_size, device, savefilepath, fxr_savename, lr)

In [ ]:
plot_losses([fxrgl, fxrdl], loss_names, epochs, fileprefix + fxr_savename + '_loss.png')

In [ ]:
# Define the initial condition and desired generated length
initial_condition = torch.tensor(fxr_tensor[:condition_size], dtype=torch.float32).to(device)
print(generated_length)

# Generate the new time series
norm_sample_fxr = generate_time_series(fxr_gen, initial_condition, generated_length, device)

In [ ]:
sample_fxr = denormalize(norm_sample_fxr, mlr_fxr, slr_fxr)
sim_returns.append(sample_fxr[:generated_length])

### Train Commodities Model

In [ ]:
csr_savename = 'csrmodel'
csr_gen, _, csrgl, csrdl = train_cgan(csr_dataset, window_size, condition_size, 
                              epochs, batch_size, device, savefilepath, csr_savename, lr)

In [ ]:
plot_losses([csrgl, csrdl], loss_names, epochs, fileprefix + csr_savename + '_loss.png')

In [ ]:
# Define the initial condition and desired generated length
initial_condition = torch.tensor(csr_tensor[:condition_size], dtype=torch.float32).to(device)

# Generate the new time series
norm_sample_csr = generate_time_series(csr_gen, initial_condition, generated_length, device)

In [ ]:
sample_csr = denormalize(norm_sample_csr, mlr_csr, slr_csr)
sim_returns.append(sample_csr[:generated_length])

In [ ]:
stock_index.index[:-1]

#### Daily Returns

In [ ]:
cganmlp_filename = fileprefix + 'Returns.png'
plot_daily_returns(sim_returns, stock_index.index[:-1], cganmlp_filename, lst_labels)

#### Run Empirical Tests

In [ ]:
sim_returns = [
    (
        lambda arr: arr[np.isfinite(arr)]
    )(
        (
            x.detach().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)
        ).astype(np.float64)
    )
    for x in sim_returns
]

In [ ]:
df = run_tests(sim_returns, lst_labels)
df = df.set_index('Label')
pd.set_option('display.precision', 2)
df

In [ ]:
df.to_latex()

#### Plot ACF for Returns and Squared Returns

In [ ]:
cgamlp_filename = fileprefix + 'ACFs.png'
acf_plots_block(sim_returns, lst_labels, cgamlp_filename)

#### QQ Plot

In [ ]:
cvae_qq_filename = fileprefix + 'QQ.png'
qq_plot(lst_returns, sim_returns, cvae_qq_filename, num_cols=2, labels=lst_labels)

#### Predictive and Discriminative Scores

In [ ]:
syn_lst = list()
for isym in sim_returns:
    syn_lst.append(real_data_loading(isym, 32))

In [ ]:
cgancnn_df = calculate_scores(real_lst, syn_lst, lst_labels, device)

In [ ]:
cgancnn_df

In [ ]:
cgancnn_df.to_latex()